## 1. 路径配置与标签加载

In [ ]:
## 1. 路径配置与标签加载
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.patches import Rectangle, Patch
import h5py
import pandas as pd
from collections import Counter
from scipy.stats import entropy

# ============================================================================
# 路径配置 - 修改此处切换被试
# ============================================================================

SUBJECT_ID = "ODP_01_qhlazec"
SPLIT_TYPE = "test"

BASE_RESULT_DIR = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/runs/loso_37fold")

if SPLIT_TYPE == "test":
    fold_candidates = list(BASE_RESULT_DIR.glob(f"*_test_{SUBJECT_ID}"))
    FOLD_DIR = fold_candidates[0] if len(fold_candidates) > 0 else BASE_RESULT_DIR / f"fold_01_test_{SUBJECT_ID}"
else:
    FOLD_DIR = BASE_RESULT_DIR / "fold_01_test_ODP_01_qhlazec"

DATA_ROOT = Path("/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/downsampling/3d")
DOWNSAMPLED_FILE = DATA_ROOT / f"{SUBJECT_ID}_downsampled.npz"
PRED_SOFTMAX_FILE = FOLD_DIR / "pred_3d" / f"{SPLIT_TYPE}_{SUBJECT_ID}_pred_softmax_3d.npz"
GT_3D_FILE = DATA_ROOT / "3d" / f"{SUBJECT_ID}_3d.npz"
LABEL_EXCEL = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi.xlsx")

OUTPUT_DIR = FOLD_DIR / "figs" / "confidence_overlay"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHANNEL_MPRAGE = 341
CHANNEL_QSM = 350

# ============================================================================
# 可视化参数
# ============================================================================

TAU = 0.4
N_CLASSES = 102
SLICE_AXIS = 'axial'
AUTO_SELECT_SLICES = True
MANUAL_SLICES = [30, 50, 70]
CONTOUR_LEVELS = [0.3]
CONTOUR_LINEWIDTH = 1.5

ZOOM_REGIONS = {
    "cortex_white": {"z": None, "x": None, "y": None, "hw": 20},
    "basal_ganglia": {"z": None, "x": None, "y": None, "hw": 20},
    "brainstem": {"z": None, "x": None, "y": None, "hw": 20}
}

print("✓ 路径配置完成")
print(f"  被试: {SUBJECT_ID}, 类型: {SPLIT_TYPE}")
print(f"  Fold: {FOLD_DIR.name}")
print(f"  输出: {OUTPUT_DIR}")

# ============================================================================
# 加载FreeSurfer标签映射
# ============================================================================

print("\n加载FreeSurfer标签映射...")
label_df = pd.read_excel(LABEL_EXCEL)
valid_labels = label_df[label_df['one_hot_loc_alex_label'] != '[]'].copy()
valid_labels['label_idx'] = valid_labels['one_hot_loc_alex_label'].astype(int)

LABEL_NAMES = {0: 'Background'}
LABEL_COLORS_RGB = {0: (0, 0, 0)}

for idx, row in valid_labels.iterrows():
    label_idx = int(row['label_idx'])
    if 1 <= label_idx <= N_CLASSES and label_idx not in LABEL_NAMES:
        LABEL_NAMES[label_idx] = row['tissue_name'].strip("'")
        LABEL_COLORS_RGB[label_idx] = (row['R']/255, row['G']/255, row['B']/255)

colors_list = [LABEL_COLORS_RGB.get(i, (0, 0, 0)) for i in range(N_CLASSES)]
cmap_freesurfer = mcolors.ListedColormap(colors_list)

print(f"✓ 加载了 {len(LABEL_NAMES)} 个标签")

In [ ]:
## 2. 加载数据并计算置信度与不确定性
# print("加载数据...")

# 预测概率
pred_data = np.load(PRED_SOFTMAX_FILE)
pred_softmax = pred_data['pred_softmax_3d']
print(f"  预测概率: {pred_softmax.shape}")

# Ground Truth
gt_data = np.load(GT_3D_FILE)
gt_proba = gt_data['proba_labels']
region_mask = gt_data['region_mask_lr']
print(f"  Ground Truth: {gt_proba.shape}")

# MPRAGE底图
if DOWNSAMPLED_FILE.exists():
    downsampled_data = np.load(DOWNSAMPLED_FILE)
    data_lr = downsampled_data['data_lr']
    anatomy_img = data_lr[..., CHANNEL_MPRAGE]
    anatomy_name = "MPRAGE"
    print(f"  MPRAGE底图: {anatomy_img.shape}")
else:
    anatomy_img = np.zeros(pred_softmax.shape[:3])
    anatomy_name = "No Anatomy"

# ============================================================================
# 计算置信度
# ============================================================================

print("\n计算置信度与不确定性...")

top2_indices = np.argsort(pred_softmax, axis=-1)[..., -2:]
top1_class = top2_indices[..., 1]
top2_class = top2_indices[..., 0]

top2_probs = np.take_along_axis(pred_softmax, top2_indices, axis=-1)
p1 = top2_probs[..., 1]
p2 = top2_probs[..., 0]

# 边际差 (margin)
margin = p1 - p2  # (Z, X, Y)
alpha_map = np.clip(margin / TAU, 0, 1)
alpha_map[region_mask == 0] = 0

# ============================================================================
# 计算熵图（不确定性）
# ============================================================================

# H(p) = -Σ_k p_k log(p_k)
# 使用scipy.stats.entropy，axis=-1计算每个体素的熵
epsilon = 1e-10  # 避免log(0)
pred_softmax_safe = np.clip(pred_softmax, epsilon, 1.0)

# entropy使用自然对数，转换为bits需要除以log(2)
entropy_map = entropy(pred_softmax_safe.T, axis=0).T  # scipy要求axis=0，所以转置
entropy_map = entropy_map / np.log(2)  # 转为bits

# 应用ROI掩码
entropy_map_masked = entropy_map.copy()
entropy_map_masked[region_mask == 0] = 0

# 统计
entropy_roi = entropy_map[region_mask > 0]
margin_roi = margin[region_mask > 0]

print(f"  边际差 (Margin, ROI内):")
print(f"    Mean={margin_roi.mean():.4f}, Median={np.median(margin_roi):.4f}")
print(f"    Range=[{margin_roi.min():.4f}, {margin_roi.max():.4f}]")

print(f"  熵 (Entropy, ROI内):")
print(f"    Mean={entropy_roi.mean():.4f} bits, Median={np.median(entropy_roi):.4f} bits")
print(f"    Range=[{entropy_roi.min():.4f}, {entropy_roi.max():.4f}] bits")
print(f"    Max possible: {np.log2(N_CLASSES):.2f} bits (uniform distribution)")

# ============================================================================
# 自动选择切片
# ============================================================================

if AUTO_SELECT_SLICES:
    print("\n自动选择切片...")
    axis_size = pred_softmax.shape[0]
    slice_scores = []
    
    for z in range(axis_size):
        mask_slice = region_mask[z] > 0
        if mask_slice.sum() > 100:
            # 使用熵的标准差作为信息量度量
            entropy_std = entropy_map[z][mask_slice].std()
            mask_ratio = mask_slice.sum() / mask_slice.size
            slice_scores.append((z, entropy_std * mask_ratio))
    
    slice_scores.sort(key=lambda x: x[1], reverse=True)
    selected_slices = []
    for z, score in slice_scores:
        if len(selected_slices) == 0 or all(abs(z - s) >= 10 for s in selected_slices):
            selected_slices.append(z)
        if len(selected_slices) >= 3:
            break
    
    MANUAL_SLICES = sorted(selected_slices)
    print(f"  选择的切片: {MANUAL_SLICES}")

print("\n✓ 数据准备完成")

In [ ]:
# ====================== 绘制三联图（换行与不重叠版） ======================
import textwrap

# 建议：关闭 constrained_layout，手动控制边距，给右侧图例留白
fig, axes = plt.subplots(1, 3, figsize=(36, 12))
fig.patch.set_facecolor('black')

# —— 工具：包一层自动换行 —— 
def wrap(s, width=28):
    # 你可以按需要调宽度（越小越容易换行）
    return textwrap.fill(s, width=width)

def set_title(ax, s, fontsize=14, pad=8):
    txt = ax.set_title(wrap(s), fontsize=fontsize, fontweight='bold',
                       color='white', pad=pad, wrap=True)
    # 增加行距，避免多行贴在一起
    txt.set_linespacing(1.15)

def plot_categorical_map(ax, label_map, mask, title, show_red_mask=False, red_mask_data=None):
    label_rgb = np.zeros((*label_map.shape, 3))
    for class_idx in range(N_CLASSES):
        class_mask = (label_map == class_idx) & (mask > 0)
        if class_mask.any():
            color = LABEL_COLORS_RGB.get(class_idx, (0, 0, 0))
            label_rgb[class_mask] = color

    ax.imshow(label_rgb, aspect='auto', interpolation='nearest')

    if show_red_mask and red_mask_data is not None:
        red_overlay = np.zeros((*label_map.shape, 4))
        red_overlay[red_mask_data] = RED_MASK_COLOR
        ax.imshow(red_overlay, aspect='auto', interpolation='nearest')

    ax.contour(mask, levels=[0.5], colors='cyan',
               linewidths=1.0, linestyles='--', alpha=0.5)
    ax.axis('off')
    set_title(ax, title, fontsize=14, pad=10)

# (a) Reference Mode
plot_categorical_map(
    axes[0],
    ref_mode,
    mask_slice,
    f'(a) Reference Mode\n(argmax of soft reference)\nSlice {SLICE_IDX}'
)

# (b) Top-1 Prediction
plot_categorical_map(
    axes[1],
    pred_top1,
    mask_slice,
    f'(b) Top-1 Prediction\n(argmax of p_pred, post-T)\nSlice {SLICE_IDX}'
)

# (c) Top-3 Union + 红遮罩
plot_categorical_map(
    axes[2],
    ref_mode,  # 用参考颜色做底色
    mask_slice,
    f'(c) Top-{K_TOP} Union + Uncovered Mask\n'
    f'Red = Ref mode NOT in pred Top-{K_TOP}\n'
    f'Coverage: {100-red_ratio:.1f}%, Uncovered: {red_ratio:.1f}%',
    show_red_mask=True,
    red_mask_data=red_mask
)

# —— 图例：放右侧，避免挤占子图 —— 
legend_labels = []
for class_idx in [0, 2, 7, 10, 41, 42]:
    if class_idx in LABEL_NAMES:
        legend_labels.append(
            Patch(facecolor=LABEL_COLORS_RGB[class_idx],
                  edgecolor='white', label=LABEL_NAMES[class_idx])
        )
legend_labels.append(
    Patch(facecolor=RED_MASK_COLOR[:3], alpha=RED_MASK_COLOR[3],
          edgecolor='white', label='Uncovered (Red Mask)')
)

fig.legend(handles=legend_labels, loc='center left', bbox_to_anchor=(0.985, 0.5),
           frameon=True, facecolor='black', edgecolor='white',
           fontsize=10, labelcolor='white')

# —— 主标题：先手动换行，再留出顶端空间 —— 
suptitle_str = (
    f'Figure 1: Reference Mode | Top-1 Prediction | Top-{K_TOP} Union with Uncovered Regions\n'
    f'Subject: {SUBJECT_ID}, Slice: {SLICE_IDX}, {SPLIT_TYPE.upper()} set'
)
fig.suptitle(textwrap.fill(suptitle_str, width=70),
             fontsize=16, fontweight='bold', color='white', y=0.98)

# —— 关键：留白，防止重叠（右侧图例+上方主标题） —— 
# right 要小于 1 给 legend，top 要小于 1 给 suptitle，wspace 调小减少子图间空隙
fig.subplots_adjust(left=0.03, right=0.93, top=0.90, bottom=0.03, wspace=0.03)

plt.show()


In [ ]:
## Figure 1: 三联图 - Reference Mode | Top-1 | Top-3 Union + 未覆盖红遮罩
# ============================================================================
# ISMRM 图像制图工程师 - Figure 1 生成
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

# ============================================================================
# 参数配置
# ============================================================================

SLICE_IDX = 17  # 可手动切换切片
TAU_HARD = True  # 使用硬规则（参考众数是否落入Top-3）
RED_MASK_COLOR = (220/255, 0, 0, 0.6)  # RGBA for red mask
K_TOP = 3  # Top-K

print(f"=== Figure 1 三联图生成 ===")
print(f"被试: {SUBJECT_ID}, 切片: {SLICE_IDX}")
print(f"规则: 硬规则（参考众数是否落入Top-{K_TOP}）")

# ============================================================================
# 提取切片数据
# ============================================================================

# 预测后验 (post-T)
p_pred_slice = pred_softmax[SLICE_IDX, :, :, :]  # (H, W, C=102)

# 软参考标签
p_ref_slice = gt_proba[SLICE_IDX, :, :, :]  # (H, W, C=102)

# ROI掩膜
mask_slice = region_mask[SLICE_IDX, :, :]  # (H, W)

# 解剖底图（可选）
anatomy_slice = anatomy_img[SLICE_IDX, :, :]

# ============================================================================
# 计算三个面板的内容
# ============================================================================

# (a) 参考众数（Reference mode）: argmax of soft reference
ref_mode = np.argmax(p_ref_slice, axis=-1)  # (H, W)

# (b) Top-1 预测
pred_top1 = np.argmax(p_pred_slice, axis=-1)  # (H, W)

# (c) Top-3 union + 红遮罩
# 获取 Top-3 classes (每个像素)
top3_indices = np.argsort(p_pred_slice, axis=-1)[..., -K_TOP:]  # (H, W, 3)

# 硬规则：检查 ref_mode 是否在 pred_top3 中
covered_mask = np.zeros(ref_mode.shape, dtype=bool)  # (H, W)
for i in range(ref_mode.shape[0]):
    for j in range(ref_mode.shape[1]):
        if mask_slice[i, j] > 0:  # 只在ROI内计算
            ref_class = ref_mode[i, j]
            pred_top3_classes = top3_indices[i, j, :]
            covered_mask[i, j] = ref_class in pred_top3_classes

# 红遮罩：ref_mode 不在 Top-3 中
red_mask = (~covered_mask) & (mask_slice > 0)  # (H, W)

# ============================================================================
# 统计信息（校验输出）
# ============================================================================

roi_pixels = mask_slice > 0
total_roi = roi_pixels.sum()
red_pixels = red_mask.sum()
red_ratio = 100 * red_pixels / total_roi if total_roi > 0 else 0

print(f"\n=== 统计信息 ===")
print(f"ROI 总像素数: {total_roi}")
print(f"红遮罩像素数: {red_pixels}")
print(f"红遮罩比例: {red_ratio:.2f}%")
print(f"Top-{K_TOP} 覆盖率: {100 - red_ratio:.2f}%")

# 软覆盖分数分布（额外信息，用于对比）
soft_coverage = np.zeros(ref_mode.shape)
for i in range(ref_mode.shape[0]):
    for j in range(ref_mode.shape[1]):
        if mask_slice[i, j] > 0:
            pred_top3_classes = top3_indices[i, j, :]
            soft_coverage[i, j] = p_ref_slice[i, j, pred_top3_classes].sum()

soft_cov_roi = soft_coverage[roi_pixels]
print(f"\n软覆盖分数（额外信息）:")
print(f"  Mean={soft_cov_roi.mean():.4f}, Median={np.median(soft_cov_roi):.4f}")
print(f"  Range=[{soft_cov_roi.min():.4f}, {soft_cov_roi.max():.4f}]")

# ============================================================================
# 绘制三联图
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(36, 12))

# 通用设置函数
def plot_categorical_map(ax, label_map, mask, title, show_red_mask=False, red_mask_data=None):
    """
    绘制分类图（统一LUT，无透明度）
    
    Parameters:
    - ax: matplotlib axis
    - label_map: (H, W) 整数标签
    - mask: (H, W) ROI掩膜
    - title: 标题
    - show_red_mask: 是否叠加红遮罩
    - red_mask_data: (H, W) 红遮罩布尔数组
    """
    # 背景黑色
    label_rgb = np.zeros((*label_map.shape, 3))
    
    # 应用FreeSurfer LUT
    for class_idx in range(N_CLASSES):
        class_mask = (label_map == class_idx) & (mask > 0)
        if class_mask.any():
            color = LABEL_COLORS_RGB.get(class_idx, (0, 0, 0))
            label_rgb[class_mask] = color
    
    # 显示分类图（无透明度）
    ax.imshow(label_rgb, aspect='auto', interpolation='nearest')
    
    # 叠加红遮罩（如果需要）
    if show_red_mask and red_mask_data is not None:
        red_overlay = np.zeros((*label_map.shape, 4))
        red_overlay[red_mask_data] = RED_MASK_COLOR
        ax.imshow(red_overlay, aspect='auto', interpolation='nearest')
    
    # ROI边界（青色虚线）
    ax.contour(mask, levels=[0.5], colors='cyan', 
               linewidths=1.0, linestyles='--', alpha=0.5)
    
    ax.set_title(title, fontsize=14, fontweight='bold', color='white', pad=10)
    ax.axis('off')

# ========================================================================
# (a) Reference Mode
# ========================================================================
plot_categorical_map(
    axes[0], 
    ref_mode, 
    mask_slice,
    f'(a) Reference Mode\n(argmax of soft reference)\nSlice {SLICE_IDX}'
)

# ========================================================================
# (b) Top-1 Prediction
# ========================================================================
plot_categorical_map(
    axes[1], 
    pred_top1, 
    mask_slice,
    f'(b) Top-1 Prediction\n(argmax of p_pred, post-T)\nSlice {SLICE_IDX}'
)

# ========================================================================
# (c) Top-3 Union + 红遮罩
# ========================================================================
plot_categorical_map(
    axes[2], 
    ref_mode,  # 显示参考颜色作为底色
    mask_slice,
    f'(c) Top-{K_TOP} Union + Uncovered Mask\n' +
    f'Red = Ref mode NOT in pred Top-{K_TOP}\n' +
    f'Coverage: {100-red_ratio:.1f}%, Uncovered: {red_ratio:.1f}%',
    show_red_mask=True,
    red_mask_data=red_mask
)

# ========================================================================
# 图例（可选，显示几个主要脑区颜色）
# ========================================================================

# 选择几个代表性标签
legend_labels = []
for class_idx in [0, 2, 7, 10, 41, 42]:  # 示例：背景+几个主要脑区
    if class_idx in LABEL_NAMES:
        legend_labels.append(
            Patch(facecolor=LABEL_COLORS_RGB[class_idx], 
                  edgecolor='white', label=LABEL_NAMES[class_idx])
        )

# 红遮罩图例
legend_labels.append(
    Patch(facecolor=RED_MASK_COLOR[:3], alpha=RED_MASK_COLOR[3],
          edgecolor='white', label='Uncovered (Red Mask)')
)

# 添加图例到右侧
fig.legend(handles=legend_labels, loc='center left', bbox_to_anchor=(1.0, 0.5),
           frameon=True, facecolor='black', edgecolor='white', 
           fontsize=10, labelcolor='white')

# ========================================================================
# 统一设置
# ========================================================================

plt.tight_layout()
fig.patch.set_facecolor('black')

# 添加总标题
fig.suptitle(f'Figure 1: Reference Mode | Top-1 Prediction | Top-{K_TOP} Union with Uncovered Regions\n' +
             f'Subject: {SUBJECT_ID}, Slice: {SLICE_IDX}, {SPLIT_TYPE.upper()} set',
             fontsize=16, fontweight='bold', color='white', y=0.98)

plt.show()

# ============================================================================
# 保存统计数据（CSV）- 可选
# ============================================================================

# 软覆盖直方图（10 bins）
hist, bin_edges = np.histogram(soft_cov_roi, bins=10, range=(0, 1))
hist_data = {
    'bin_left': bin_edges[:-1],
    'bin_right': bin_edges[1:],
    'count': hist,
    'percentage': 100 * hist / total_roi
}

print(f"\n=== 软覆盖分数直方图 (10 bins) ===")
for i in range(10):
    print(f"  [{hist_data['bin_left'][i]:.2f}, {hist_data['bin_right'][i]:.2f}): " +
          f"{hist_data['count'][i]} pixels ({hist_data['percentage'][i]:.2f}%)")

print(f"\n✓ Figure 1 生成完成！")

In [ ]:
# ====================== 四联图：无总标题，面板内标题 + 右侧全局色条 ======================
import numpy as np
import matplotlib.pyplot as plt
import textwrap
from matplotlib.patches import Patch

# ---------- 配置：熵单位 ----------
# 选 "bits" 或 "nats"
ENTROPY_UNIT = "bits"

# ---------- 工具函数 ----------
def panel_title(ax, s):
    s = textwrap.fill(s, width=24)
    ax.text(
        0.01, 0.99, s,
        transform=ax.transAxes, va='top', ha='left',
        fontsize=13, fontweight='bold', color='white',
        bbox=dict(boxstyle='round,pad=0.3', facecolor=(0,0,0,0.55), edgecolor='none')
    )

def plot_categorical_map(ax, label_map, mask, title, show_red_mask=False, red_mask_data=None):
    label_rgb = np.zeros((*label_map.shape, 3))
    for class_idx in range(N_CLASSES):
        class_mask = (label_map == class_idx) & (mask > 0)
        if class_mask.any():
            label_rgb[class_mask] = LABEL_COLORS_RGB.get(class_idx, (0, 0, 0))
    ax.imshow(label_rgb, aspect='auto', interpolation='nearest')

    if show_red_mask and red_mask_data is not None:
        red_overlay = np.zeros((*label_map.shape, 4))
        red_overlay[red_mask_data] = RED_MASK_COLOR
        ax.imshow(red_overlay, aspect='auto', interpolation='nearest')

    ax.contour(mask, levels=[0.5], colors='cyan', linewidths=1.0, linestyles='--', alpha=0.5)
    ax.axis('off')
    panel_title(ax, title)

# ---------- 画布 ----------
fig, axes = plt.subplots(1, 4, figsize=(46, 12), constrained_layout=False)
fig.patch.set_facecolor('black')

# ---------- (a) Groundtruth ----------
plot_categorical_map(
    axes[0], ref_mode, mask_slice,
    '(a) Subject-specific parcellation (Groundtruth)'
)

# ---------- (b) Top-1 ----------
plot_categorical_map(
    axes[1], pred_top1, mask_slice,
    '(b) Top-1 prediction (post-T)'
)

# ---------- (c) Top-3 ∪ + 红遮罩 ----------
coverage = 100.0 - float(red_ratio)
missed = float(red_ratio)
plot_categorical_map(
    axes[2], ref_mode, mask_slice,
    f'(c) Top-3 union (GT∈Top-3 → GT color; else red).\n'
    f'Top-3 coverage = {coverage:.1f}% (missed = {missed:.1f}%)',
    show_red_mask=True, red_mask_data=red_mask
)

# ---------- (d) Uncertainty: Margin + Entropy ----------
# 若已有 margin / entropy_map，可直接替换计算段
p_sorted = np.sort(p_pred_slice, axis=-1)
p1, p2 = p_sorted[..., -1], p_sorted[..., -2]
margin_map = p1 - p2

eps = 1e-12
p_safe = np.clip(p_pred_slice, eps, 1.0)
if ENTROPY_UNIT.lower() == "bits":
    entropy_map = -np.sum(p_safe * (np.log(p_safe) / np.log(2.0)), axis=-1)
    entropy_unit_label = "bits"
else:  # nats
    entropy_map = -np.sum(p_safe * np.log(p_safe), axis=-1)
    entropy_unit_label = "nats"

roi = mask_slice > 0
if roi.any():
    margin_min, margin_max = margin_map[roi].min(), margin_map[roi].max()
    margin_norm = (margin_map - margin_min) / (margin_max - margin_min + 1e-8)
    margin_norm[~roi] = 0.0
    ent_roi = entropy_map[roi]
    ent_vmax = np.percentile(ent_roi, 95)
    hi_thr = np.percentile(ent_roi, 75)
else:
    margin_norm = np.zeros_like(margin_map)
    ent_vmax = entropy_map.max() if np.isfinite(entropy_map.max()) else 1.0
    hi_thr = 0.0

axes[3].imshow(margin_norm, cmap='gray', aspect='auto', interpolation='bilinear', vmin=0, vmax=1)
entropy_masked = np.ma.masked_where(~roi, entropy_map)
im = axes[3].imshow(entropy_masked, cmap='hot', aspect='auto', interpolation='bilinear',
                    alpha=0.70, vmin=0.0, vmax=ent_vmax)

axes[3].contour(mask_slice, levels=[0.5], colors='cyan', linewidths=1.0, linestyles='--', alpha=0.5)
if roi.any():
    axes[3].contour(entropy_map, levels=[hi_thr], colors='yellow', linewidths=2.0, alpha=0.85)

def panel_title(ax, s):
    """面板内标题：逐段换行，保留手动换行与空行"""
    lines = s.split('\n')  # 手动分段
    wrapped = [textwrap.fill(line, width=24) if line.strip() != '' else '' for line in lines]
    s2 = '\n'.join(wrapped)  # 保留空行 => 两个 \n 连在一起
    ax.text(
        0.01, 0.99, s2,
        transform=ax.transAxes, va='top', ha='left',
        fontsize=13, fontweight='bold', color='white',
        bbox=dict(boxstyle='round,pad=0.3', facecolor=(0,0,0,0.55), edgecolor='none'),
        multialignment='left'
    )


panel_title(
    axes[3],
    '(d) Uncertainty\n'
    'Background = Margin (p1 - p2)\n'
    f'Overlay = Entropy H(p) [{entropy_unit_label}]'
)

axes[3].axis('off')

# ---------- 右侧图例 ----------
legend_handles = []
for class_idx in [0, 2, 7, 10, 41, 42]:  # 示例：可根据需要调整
    if class_idx in LABEL_NAMES:
        legend_handles.append(
            Patch(facecolor=LABEL_COLORS_RGB[class_idx], edgecolor='white',
                  label=LABEL_NAMES[class_idx])
        )
legend_handles.append(
    Patch(facecolor=RED_MASK_COLOR[:3], alpha=RED_MASK_COLOR[3],
          edgecolor='white', label='Uncovered (Red Mask)')
)

# ---------- 布局留白 + 右侧全局色条 ----------
fig.subplots_adjust(left=0.02, right=0.90, top=0.96, bottom=0.03, wspace=0.03)

cax = fig.add_axes([0.92, 0.15, 0.015, 0.70])  # [left, bottom, width, height] in figure coords
cbar = plt.colorbar(im, cax=cax)
cbar.set_label(f'Entropy ({entropy_unit_label})', rotation=270, labelpad=14, color='white', fontsize=10)
cbar.ax.tick_params(colors='white', labelsize=9)

fig.legend(
    handles=legend_handles,
    loc='center left',
    bbox_to_anchor=(0.975, 0.5),
    frameon=True, facecolor='black', edgecolor='white',
    fontsize=10, labelcolor='white'
)

plt.show()


## Figure 4: Boundary vs Interior Analysis (post-T)

**目标**: 比较边界区域（≤2mm from GT boundaries）与内部区域（≥3mm）的 Top-3 coverage 和 soft-ECE（post-T）。

**输入数据** (per fold):
- `pred_probs`: (Z, Y, X, C) 或 (N, C) - post-T 预测后验概率
- `soft_ref`: (Z, Y, X, C) 或 (N, C) - 软参考标签（blur→downsample→renorm）
- `seg_labels`: (Z, Y, X) - FreeSurfer 离散标签（仅用于边界计算）
- Voxel spacing: (1.8, 1.8, 3.0) mm

**方法**:
1. 使用 6-connectivity 从 seg_labels 计算边界
2. 距离变换定义 Edge (≤2mm) 和 Interior (≥3mm) 掩膜
3. 计算 soft Top-3 coverage 和 soft-ECE (post-T)
4. 跨所有 folds 收集统计数据
5. 绘制双面板 violin/box 图

In [ ]:
## Figure 4: Boundary vs Interior Analysis - 修复版（放宽概率和检查）

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import distance_transform_edt, binary_erosion, binary_dilation
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 核心函数定义（修复版）
# ============================================================================

def compute_boundary(seg_labels):
    """
    计算边界掩膜（6-connectivity）
    
    Parameters:
    - seg_labels: (Z, X, Y) 整数标签数组
    
    Returns:
    - boundary: (Z, X, Y) 布尔数组，True=边界体素
    """
    # 排除背景（label=0）
    foreground = seg_labels > 0
    
    # 6-connectivity: 检查每个体素的6邻域是否有不同标签
    boundary = np.zeros_like(seg_labels, dtype=bool)
    for dz, dx, dy in [(-1,0,0), (1,0,0), (0,-1,0), (0,1,0), (0,0,-1), (0,0,1)]:
        shifted = np.roll(seg_labels, shift=(dz, dx, dy), axis=(0, 1, 2))
        boundary |= (seg_labels != shifted) & foreground
    
    return boundary


def distance_mm_from_boundary(seg_labels, voxel_spacing=(1.8, 1.8, 3.0)):
    """
    计算每个体素到最近边界的欧氏距离（mm）
    
    Parameters:
    - seg_labels: (Z, X, Y) 整数标签
    - voxel_spacing: (sz, sx, sy) 体素尺寸（mm）
    
    Returns:
    - dist_mm: (Z, X, Y) 距离图（mm）
    """
    boundary = compute_boundary(seg_labels)
    
    # 距离变换：计算到边界的距离（单位：体素）
    dist_voxels = distance_transform_edt(~boundary, sampling=voxel_spacing)
    
    return dist_voxels


def soft_topk_coverage(pred_probs, soft_ref, k=3, mask=None):
    """
    计算 soft Top-K coverage (修复版：自动重新归一化)
    
    Parameters:
    - pred_probs: (N, C) 或 (Z, X, Y, C) 预测后验
    - soft_ref: (N, C) 或 (Z, X, Y, C) 软参考标签
    - k: Top-K
    - mask: (N,) 或 (Z, X, Y) 布尔掩膜，仅在 mask=True 处计算
    
    Returns:
    - coverage_scores: (N,) 每个体素的覆盖分数
    """
    # 展平为 2D
    if pred_probs.ndim == 4:
        Z, X, Y, C = pred_probs.shape
        pred_probs = pred_probs.reshape(-1, C).copy()  # copy to avoid modifying original
        soft_ref = soft_ref.reshape(-1, C)
        if mask is not None:
            mask = mask.reshape(-1)
    else:
        pred_probs = pred_probs.copy()
    
    N, C = pred_probs.shape
    
    # 数据校验
    assert pred_probs.shape == soft_ref.shape, f"Shape mismatch! pred={pred_probs.shape}, ref={soft_ref.shape}"
    assert np.all((pred_probs >= -1e-6) & (pred_probs <= 1 + 1e-6)), "Probs out of [0,1]!"
    
    # 检查概率和，如果不等于1则重新归一化
    prob_sums = pred_probs.sum(axis=1)
    max_deviation = np.abs(prob_sums - 1.0).max()
    
    if max_deviation > 1e-3:
        print(f"    ⚠ 概率和偏离 1.0: max_deviation={max_deviation:.6f}，重新归一化...")
        # 重新归一化
        pred_probs = pred_probs / (prob_sums[:, np.newaxis] + 1e-10)
    
    # 裁剪到 [0, 1]
    pred_probs = np.clip(pred_probs, 0, 1)
    
    # 获取 Top-K 索引
    topk_indices = np.argpartition(pred_probs, -k, axis=1)[:, -k:]  # (N, k)
    
    # 计算覆盖分数
    coverage = np.zeros(N)
    for i in range(N):
        if mask is None or mask[i]:
            coverage[i] = soft_ref[i, topk_indices[i]].sum()
    
    # 应用掩膜
    if mask is not None:
        coverage = coverage[mask]
    
    return coverage


def soft_ece(pred_probs, soft_ref, n_bins=15, mask=None):
    """
    计算 soft Expected Calibration Error (ECE) - 修复版
    
    Parameters:
    - pred_probs: (N, C) 预测后验
    - soft_ref: (N, C) 软参考标签
    - n_bins: 置信度分箱数
    - mask: (N,) 布尔掩膜
    
    Returns:
    - ece: 标量 ECE 值
    """
    # 展平
    if pred_probs.ndim == 4:
        pred_probs = pred_probs.reshape(-1, pred_probs.shape[-1]).copy()
        soft_ref = soft_ref.reshape(-1, soft_ref.shape[-1])
        if mask is not None:
            mask = mask.reshape(-1)
    else:
        pred_probs = pred_probs.copy()
    
    N = pred_probs.shape[0]
    
    # 应用掩膜
    if mask is not None:
        pred_probs = pred_probs[mask]
        soft_ref = soft_ref[mask]
        N = pred_probs.shape[0]
    
    if N == 0:
        return np.nan
    
    # 重新归一化（如果需要）
    prob_sums = pred_probs.sum(axis=1)
    if np.abs(prob_sums - 1.0).max() > 1e-3:
        pred_probs = pred_probs / (prob_sums[:, np.newaxis] + 1e-10)
    
    # 裁剪
    pred_probs = np.clip(pred_probs, 0, 1)
    
    # 置信度 = max(pred_probs)
    conf = pred_probs.max(axis=1)  # (N,)
    
    # 预测类别
    pred_class = pred_probs.argmax(axis=1)  # (N,)
    
    # "正确概率" = soft_ref 在预测类别上的值
    p_correct = soft_ref[np.arange(N), pred_class]  # (N,)
    
    # 等宽分箱
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    
    for b in range(n_bins):
        bin_mask = (conf >= bin_edges[b]) & (conf < bin_edges[b + 1])
        if b == n_bins - 1:  # 最后一个bin包含右端点
            bin_mask = (conf >= bin_edges[b]) & (conf <= bin_edges[b + 1])
        
        n_b = bin_mask.sum()
        if n_b > 0:
            mean_conf = conf[bin_mask].mean()
            mean_correct = p_correct[bin_mask].mean()
            ece += np.abs(mean_conf - mean_correct) * (n_b / N)
    
    return ece


# ============================================================================
# 主分析流程
# ============================================================================

print("=" * 80)
print("Figure 4: Boundary vs Interior Analysis (post-T) - 修复版")
print("=" * 80)

# 配置参数
VOXEL_SPACING = (3.0, 1.8, 1.8)  # (Z, X, Y) mm - 注意：匹配数据轴顺序
EDGE_THRESHOLD = 2.0  # mm
INTERIOR_THRESHOLD = 3.0  # mm
K_TOP = 3
N_BINS_ECE = 15

# 查找所有 folds
BASE_RESULT_DIR = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/runs/loso_37fold")
DATA_ROOT = Path("/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/downsampling/3d")

# 获取所有 fold 目录（假设命名格式为 fold_XX_test_SUBJECT 或类似）
fold_dirs = sorted(list(BASE_RESULT_DIR.glob("fold_*_test_*")))

print(f"\n找到 {len(fold_dirs)} 个 folds")
print(f"体素尺寸: {VOXEL_SPACING} mm (Z, X, Y)")
print(f"边界定义: ≤ {EDGE_THRESHOLD} mm")
print(f"内部定义: ≥ {INTERIOR_THRESHOLD} mm")
print(f"Top-K: {K_TOP}")
print(f"ECE bins: {N_BINS_ECE}\n")

# 收集所有 folds 的指标
results = {
    'edge_coverage': [],
    'interior_coverage': [],
    'edge_ece': [],
    'interior_ece': []
}

# 循环处理每个 fold
for fold_idx, fold_dir in enumerate(fold_dirs[:10]):  # 先测试前10个folds
    fold_name = fold_dir.name
    print(f"[{fold_idx + 1}/{len(fold_dirs[:10])}] Processing: {fold_name}")
    
    try:
        # 提取被试ID（假设格式为 fold_XX_test_SUBJECTID）
        parts = fold_name.split('_test_')
        if len(parts) == 2:
            subject_id = parts[1]
        else:
            print(f"  ⚠ 无法解析被试ID，跳过")
            continue
        
        # 构建文件路径
        pred_file = fold_dir / "pred_3d" / f"test_{subject_id}_pred_softmax_3d.npz"
        gt_file = DATA_ROOT / "3d" / f"{subject_id}_3d.npz"
        
        if not pred_file.exists():
            print(f"  ⚠ 预测文件不存在: {pred_file.name}")
            continue
        if not gt_file.exists():
            print(f"  ⚠ GT文件不存在: {gt_file.name}")
            continue
        
        # 加载数据
        pred_data = np.load(pred_file)
        gt_data = np.load(gt_file)
        
        pred_probs = pred_data['pred_softmax_3d']  # (Z, X, Y, C)
        soft_ref = gt_data['proba_labels']  # (Z, X, Y, C)
        region_mask = gt_data['region_mask_lr']  # (Z, X, Y)
        
        # 生成 seg_labels（argmax of soft_ref，用于边界计算）
        seg_labels = np.argmax(soft_ref, axis=-1).astype(np.int32)  # (Z, X, Y)
        
        print(f"  Shapes: pred={pred_probs.shape}, ref={soft_ref.shape}, mask={region_mask.shape}")
        
        # 计算距离图
        dist_mm = distance_mm_from_boundary(seg_labels, voxel_spacing=VOXEL_SPACING)
        
        # 定义 Edge 和 Interior 掩膜（排除背景）
        foreground = region_mask > 0
        edge_mask = (dist_mm <= EDGE_THRESHOLD) & foreground
        interior_mask = (dist_mm >= INTERIOR_THRESHOLD) & foreground
        
        n_edge = edge_mask.sum()
        n_interior = interior_mask.sum()
        
        print(f"  Edge voxels: {n_edge}, Interior voxels: {n_interior}")
        
        if n_edge == 0 or n_interior == 0:
            print(f"  ⚠ Edge 或 Interior 为空，跳过")
            continue
        
        # 计算 Top-3 Coverage
        edge_cov = soft_topk_coverage(pred_probs, soft_ref, k=K_TOP, mask=edge_mask)
        interior_cov = soft_topk_coverage(pred_probs, soft_ref, k=K_TOP, mask=interior_mask)
        
        # 计算 soft-ECE
        edge_ece_val = soft_ece(pred_probs, soft_ref, n_bins=N_BINS_ECE, mask=edge_mask)
        interior_ece_val = soft_ece(pred_probs, soft_ref, n_bins=N_BINS_ECE, mask=interior_mask)
        
        # 存储结果（每个fold一个均值）
        results['edge_coverage'].append(edge_cov.mean())
        results['interior_coverage'].append(interior_cov.mean())
        results['edge_ece'].append(edge_ece_val)
        results['interior_ece'].append(interior_ece_val)
        
        print(f"  ✓ Top-3 cov: Edge={edge_cov.mean():.4f}, Interior={interior_cov.mean():.4f}")
        print(f"  ✓ Soft-ECE: Edge={edge_ece_val:.4f}, Interior={interior_ece_val:.4f}\n")
        
    except Exception as e:
        import traceback
        print(f"  ✗ 错误: {e}")
        print(f"  {traceback.format_exc()}\n")
        continue

# 转为数组
for key in results:
    results[key] = np.array(results[key])

print("=" * 80)
print(f"完成！共处理 {len(results['edge_coverage'])} 个有效 folds")
print("=" * 80)

## 可选：保存图像和数据

如果需要保存高分辨率图像和统计数据，请运行下面的代码：

```python
# 保存 Figure 4（600 dpi PNG + PDF）
output_file_png = OUTPUT_DIR / 'Fig4_boundary_analysis_postT.png'
output_file_pdf = OUTPUT_DIR / 'Fig4_boundary_analysis_postT.pdf'

# 重新绘制并保存
fig.savefig(output_file_png, dpi=600, bbox_inches='tight', facecolor='white')
fig.savefig(output_file_pdf, bbox_inches='tight', facecolor='white')

print(f"✓ 已保存: {output_file_png}")
print(f"✓ 已保存: {output_file_pdf}")

# 保存统计数据到 CSV
import pandas as pd

stats_df = pd.DataFrame({
    'fold_idx': range(len(results['edge_coverage'])),
    'edge_coverage': results['edge_coverage'],
    'interior_coverage': results['interior_coverage'],
    'edge_ece': results['edge_ece'],
    'interior_ece': results['interior_ece']
})

stats_file = OUTPUT_DIR / 'Fig4_boundary_stats.csv'
stats_df.to_csv(stats_file, index=False)
print(f"✓ 已保存统计数据: {stats_file}")
```

---

## 代码说明

### 核心函数

1. **`compute_boundary(seg_labels)`**
   - 使用 6-connectivity 计算边界体素
   - 边界定义：至少一个面相邻体素有不同标签

2. **`distance_mm_from_boundary(seg_labels, voxel_spacing)`**
   - 使用欧氏距离变换计算到边界的距离（mm）
   - 考虑各向异性体素尺寸 (1.8×1.8×3.0 mm)

3. **`soft_topk_coverage(pred_probs, soft_ref, k=3, mask)`**
   - 计算 Top-K coverage：软参考在预测 Top-K 类别上的概率质量
   - 返回每个体素的覆盖分数

4. **`soft_ece(pred_probs, soft_ref, n_bins=15, mask)`**
   - 计算 soft Expected Calibration Error
   - 置信度 = max(pred_probs)
   - 正确概率 = soft_ref[预测类别]
   - ECE = Σ |avg_conf - avg_correct| × (n_bin/N)

### 区域定义

- **Edge (边界区域)**: 距离边界 ≤ 2.0 mm
- **Interior (内部区域)**: 距离边界 ≥ 3.0 mm
- **排除**: 背景体素 (label=0)

### 预期结果

根据部分体积效应和可解释性理论：
- **Top-3 coverage**: Edge 和 Interior 都应该很高（>0.9）
- **Soft-ECE**: Edge 略高于 Interior（边界区域校准难度更大）

---

## 术语对照

按照您的要求，代码中使用以下术语：
- ✅ **"anti-aliased"** (抗混叠) - 替代 "PSF-matched"
- ✅ **"post-T"** (温度缩放后) - 明确标注
- ✅ **"soft references"** (软参考) - 来自 blur→downsample→renorm pipeline
- ✅ **"6-connectivity"** (6连通) - 仅面相邻
- ✅ **"partial-volume effects"** (部分体积效应) - 边界区域的物理现象

---

In [ ]:
## 绘制 Figure 4 + 统计分析

# ============================================================================
# 统计摘要
# ============================================================================

print("\n" + "=" * 80)
print("统计摘要 (Median [IQR])")
print("=" * 80)

def print_stats(data, name):
    """打印中位数和IQR"""
    median = np.median(data)
    q25 = np.percentile(data, 25)
    q75 = np.percentile(data, 75)
    iqr = q75 - q25
    print(f"{name:30s}: {median:.4f} [{q25:.4f}, {q75:.4f}] (IQR={iqr:.4f})")
    return median, q25, q75

print("\n【Top-3 Coverage】")
edge_cov_med, edge_cov_q25, edge_cov_q75 = print_stats(results['edge_coverage'], "  Edge (≤2mm)")
int_cov_med, int_cov_q25, int_cov_q75 = print_stats(results['interior_coverage'], "  Interior (≥3mm)")

print("\n【Soft-ECE (post-T)】")
edge_ece_med, edge_ece_q25, edge_ece_q75 = print_stats(results['edge_ece'], "  Edge (≤2mm)")
int_ece_med, int_ece_q25, int_ece_q75 = print_stats(results['interior_ece'], "  Interior (≥3mm)")

# ============================================================================
# Wilcoxon 符号秩检验 (Edge vs Interior)
# ============================================================================

print("\n" + "=" * 80)
print("Wilcoxon Signed-Rank Test (Edge vs Interior)")
print("=" * 80)

try:
    # Top-3 Coverage
    stat_cov, p_cov = wilcoxon(results['edge_coverage'], results['interior_coverage'])
    print(f"Top-3 Coverage:  W={stat_cov:.2f}, p={p_cov:.4e}")
    
    # Soft-ECE
    stat_ece, p_ece = wilcoxon(results['edge_ece'], results['interior_ece'])
    print(f"Soft-ECE:        W={stat_ece:.2f}, p={p_ece:.4e}")
except Exception as e:
    print(f"⚠ Wilcoxon 检验失败: {e}")

# ============================================================================
# 绘制双面板图
# ============================================================================

print("\n" + "=" * 80)
print("生成 Figure 4...")
print("=" * 80)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')

# 准备数据
data_coverage = [results['edge_coverage'], results['interior_coverage']]
data_ece = [results['edge_ece'], results['interior_ece']]

labels = ['Edge\n(≤2mm)', 'Interior\n(≥3mm)']
colors = ['#FF6B6B', '#4ECDC4']  # 红色=边界，青色=内部

# ========================================================================
# 左图: Top-3 Coverage
# ========================================================================

parts_cov = axes[0].violinplot(data_coverage, positions=[1, 2], showmeans=False, 
                                showmedians=True, widths=0.6)

# 着色
for i, pc in enumerate(parts_cov['bodies']):
    pc.set_facecolor(colors[i])
    pc.set_alpha(0.7)
    pc.set_edgecolor('black')
    pc.set_linewidth(1.5)

# 中位数线加粗
parts_cov['cmedians'].set_color('black')
parts_cov['cmedians'].set_linewidth(2.5)

# 添加箱线图元素
bp_cov = axes[0].boxplot(data_coverage, positions=[1, 2], widths=0.3, 
                          patch_artist=True, showfliers=False,
                          boxprops=dict(facecolor='none', edgecolor='black', linewidth=1.5),
                          whiskerprops=dict(color='black', linewidth=1.5),
                          capprops=dict(color='black', linewidth=1.5),
                          medianprops=dict(color='darkred', linewidth=2.5))

# 标注中位数
axes[0].text(1, edge_cov_med + 0.02, f'{edge_cov_med:.3f}', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].text(2, int_cov_med + 0.02, f'{int_cov_med:.3f}', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[0].set_ylabel('Top-3 Coverage', fontsize=13, fontweight='bold')
axes[0].set_xticks([1, 2])
axes[0].set_xticklabels(labels, fontsize=11)
axes[0].set_ylim([0, 1])
axes[0].grid(axis='y', alpha=0.3, linestyle='--')
axes[0].set_title('(a) Top-3 Coverage (post-T)', fontsize=14, fontweight='bold', pad=10)

# ========================================================================
# 右图: Soft-ECE
# ========================================================================

parts_ece = axes[1].violinplot(data_ece, positions=[1, 2], showmeans=False, 
                                showmedians=True, widths=0.6)

# 着色
for i, pc in enumerate(parts_ece['bodies']):
    pc.set_facecolor(colors[i])
    pc.set_alpha(0.7)
    pc.set_edgecolor('black')
    pc.set_linewidth(1.5)

parts_ece['cmedians'].set_color('black')
parts_ece['cmedians'].set_linewidth(2.5)

bp_ece = axes[1].boxplot(data_ece, positions=[1, 2], widths=0.3, 
                          patch_artist=True, showfliers=False,
                          boxprops=dict(facecolor='none', edgecolor='black', linewidth=1.5),
                          whiskerprops=dict(color='black', linewidth=1.5),
                          capprops=dict(color='black', linewidth=1.5),
                          medianprops=dict(color='darkred', linewidth=2.5))

# 标注中位数
axes[1].text(1, edge_ece_med + 0.002, f'{edge_ece_med:.4f}', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].text(2, int_ece_med + 0.002, f'{int_ece_med:.4f}', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')

axes[1].set_ylabel('Soft-ECE', fontsize=13, fontweight='bold')
axes[1].set_xticks([1, 2])
axes[1].set_xticklabels(labels, fontsize=11)
axes[1].set_ylim([0, max(data_ece[0].max(), data_ece[1].max()) * 1.15])
axes[1].grid(axis='y', alpha=0.3, linestyle='--')
axes[1].set_title('(b) Soft-ECE (post-T)', fontsize=14, fontweight='bold', pad=10)

# ========================================================================
# 总标题
# ========================================================================

fig.suptitle(f'Figure 4: Boundary vs Interior Analysis (post-T, n={len(results["edge_coverage"])} folds)',
             fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout()
plt.show()

# ============================================================================
# 图注文本
# ============================================================================

caption = (
    "Fig. 4. Boundary analysis (post-T). At cortical–WM interfaces (≤2 mm), "
    "Top-3 coverage is high while soft-ECE is modestly higher than in interiors (≥3 mm), "
    "consistent with partial-volume effects and explainable Top-K behavior. "
    "Each fold represents one held-out subject in LOSO cross-validation. "
    "Violin plots show distribution with overlaid box plots (median as thick line). "
    "Evaluation uses anti-aliased soft references from the downsampling pipeline."
)

print("\n" + "=" * 80)
print("图注 (Caption)")
print("=" * 80)
print(caption)

print("\n✓ Figure 4 生成完成！")

In [ ]:
## Figure 4: Boundary vs Interior Analysis - 完整实现

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import distance_transform_edt, binary_erosion, binary_dilation
from scipy.stats import wilcoxon
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# 核心函数定义
# ============================================================================

def compute_boundary(seg_labels):
    """
    计算边界掩膜（6-connectivity）
    
    Parameters:
    - seg_labels: (Z, X, Y) 整数标签数组
    
    Returns:
    - boundary: (Z, X, Y) 布尔数组，True=边界体素
    """
    # 排除背景（label=0）
    foreground = seg_labels > 0
    
    # 6-connectivity 结构元素（仅面相邻，不含边和角）
    struct = np.zeros((3, 3, 3), dtype=bool)
    struct[1, 1, :] = True  # Z轴方向
    struct[1, :, 1] = True  # X轴方向
    struct[:, 1, 1] = True  # Y轴方向
    
    # 腐蚀：如果所有6个邻居都是相同标签，则不是边界
    eroded = binary_erosion(foreground, structure=struct)
    
    # 边界 = 前景 - 腐蚀区域
    boundary = foreground & (~eroded)
    
    # 或者：检查每个体素的6邻域是否有不同标签
    # 这里用更快的形态学方法
    boundary_alt = np.zeros_like(seg_labels, dtype=bool)
    for dz, dx, dy in [(-1,0,0), (1,0,0), (0,-1,0), (0,1,0), (0,0,-1), (0,0,1)]:
        shifted = np.roll(seg_labels, shift=(dz, dx, dy), axis=(0, 1, 2))
        boundary_alt |= (seg_labels != shifted) & foreground
    
    return boundary_alt


def distance_mm_from_boundary(seg_labels, voxel_spacing=(1.8, 1.8, 3.0)):
    """
    计算每个体素到最近边界的欧氏距离（mm）
    
    Parameters:
    - seg_labels: (Z, X, Y) 整数标签
    - voxel_spacing: (sz, sx, sy) 体素尺寸（mm）
    
    Returns:
    - dist_mm: (Z, X, Y) 距离图（mm）
    """
    boundary = compute_boundary(seg_labels)
    
    # 距离变换：计算到边界的距离（单位：体素）
    # ~boundary: 非边界区域计算距离
    dist_voxels = distance_transform_edt(~boundary, sampling=voxel_spacing)
    
    return dist_voxels


def soft_topk_coverage(pred_probs, soft_ref, k=3, mask=None):
    """
    计算 soft Top-K coverage
    
    Parameters:
    - pred_probs: (N, C) 或 (Z, X, Y, C) 预测后验
    - soft_ref: (N, C) 或 (Z, X, Y, C) 软参考标签
    - k: Top-K
    - mask: (N,) 或 (Z, X, Y) 布尔掩膜，仅在 mask=True 处计算
    
    Returns:
    - coverage_scores: (N,) 每个体素的覆盖分数
    """
    # 展平为 2D
    if pred_probs.ndim == 4:
        Z, X, Y, C = pred_probs.shape
        pred_probs = pred_probs.reshape(-1, C)
        soft_ref = soft_ref.reshape(-1, C)
        if mask is not None:
            mask = mask.reshape(-1)
    
    N, C = pred_probs.shape
    
    # 数据校验
    assert pred_probs.shape == soft_ref.shape, "Shape mismatch!"
    assert np.all((pred_probs >= 0) & (pred_probs <= 1)), "Probs out of [0,1]!"
    assert np.all(np.abs(pred_probs.sum(axis=1) - 1.0) < 1e-4), "Probs don't sum to 1!"
    
    # 获取 Top-K 索引
    topk_indices = np.argpartition(pred_probs, -k, axis=1)[:, -k:]  # (N, k)
    
    # 计算覆盖分数
    coverage = np.zeros(N)
    for i in range(N):
        if mask is None or mask[i]:
            coverage[i] = soft_ref[i, topk_indices[i]].sum()
    
    # 应用掩膜
    if mask is not None:
        coverage = coverage[mask]
    
    return coverage


def soft_ece(pred_probs, soft_ref, n_bins=15, mask=None):
    """
    计算 soft Expected Calibration Error (ECE)
    
    Parameters:
    - pred_probs: (N, C) 预测后验
    - soft_ref: (N, C) 软参考标签
    - n_bins: 置信度分箱数
    - mask: (N,) 布尔掩膜
    
    Returns:
    - ece: 标量 ECE 值
    """
    # 展平
    if pred_probs.ndim == 4:
        pred_probs = pred_probs.reshape(-1, pred_probs.shape[-1])
        soft_ref = soft_ref.reshape(-1, soft_ref.shape[-1])
        if mask is not None:
            mask = mask.reshape(-1)
    
    N = pred_probs.shape[0]
    
    # 应用掩膜
    if mask is not None:
        pred_probs = pred_probs[mask]
        soft_ref = soft_ref[mask]
        N = pred_probs.shape[0]
    
    if N == 0:
        return np.nan
    
    # 置信度 = max(pred_probs)
    conf = pred_probs.max(axis=1)  # (N,)
    
    # 预测类别
    pred_class = pred_probs.argmax(axis=1)  # (N,)
    
    # "正确概率" = soft_ref 在预测类别上的值
    p_correct = soft_ref[np.arange(N), pred_class]  # (N,)
    
    # 等宽分箱
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    
    for b in range(n_bins):
        bin_mask = (conf >= bin_edges[b]) & (conf < bin_edges[b + 1])
        if b == n_bins - 1:  # 最后一个bin包含右端点
            bin_mask = (conf >= bin_edges[b]) & (conf <= bin_edges[b + 1])
        
        n_b = bin_mask.sum()
        if n_b > 0:
            mean_conf = conf[bin_mask].mean()
            mean_correct = p_correct[bin_mask].mean()
            ece += np.abs(mean_conf - mean_correct) * (n_b / N)
    
    return ece


# ============================================================================
# 主分析流程
# ============================================================================

print("=" * 80)
print("Figure 4: Boundary vs Interior Analysis (post-T)")
print("=" * 80)

# 配置参数
VOXEL_SPACING = (3.0, 1.8, 1.8)  # (Z, X, Y) mm - 注意：匹配数据轴顺序
EDGE_THRESHOLD = 2.0  # mm
INTERIOR_THRESHOLD = 3.0  # mm
K_TOP = 3
N_BINS_ECE = 15

# 查找所有 folds
BASE_RESULT_DIR = Path("/home/jovyan/gpu_space/workspace_jiayi/KAN-git/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/runs/loso_37fold")
DATA_ROOT = Path("/home/jovyan/gpu_space/workspace_jiayi/alex_datasets/downsampling/3d")

# 获取所有 fold 目录（假设命名格式为 fold_XX_test_SUBJECT 或类似）
fold_dirs = sorted(list(BASE_RESULT_DIR.glob("fold_*_test_*")))

print(f"\n找到 {len(fold_dirs)} 个 folds")
print(f"体素尺寸: {VOXEL_SPACING} mm (Z, X, Y)")
print(f"边界定义: ≤ {EDGE_THRESHOLD} mm")
print(f"内部定义: ≥ {INTERIOR_THRESHOLD} mm")
print(f"Top-K: {K_TOP}")
print(f"ECE bins: {N_BINS_ECE}\n")

# 收集所有 folds 的指标
results = {
    'edge_coverage': [],
    'interior_coverage': [],
    'edge_ece': [],
    'interior_ece': []
}

# 循环处理每个 fold
for fold_idx, fold_dir in enumerate(fold_dirs[:5]):  # 先测试前5个folds
    fold_name = fold_dir.name
    print(f"[{fold_idx + 1}/{len(fold_dirs[:5])}] Processing: {fold_name}")
    
    try:
        # 提取被试ID（假设格式为 fold_XX_test_SUBJECTID）
        parts = fold_name.split('_test_')
        if len(parts) == 2:
            subject_id = parts[1]
        else:
            print(f"  ⚠ 无法解析被试ID，跳过")
            continue
        
        # 构建文件路径
        pred_file = fold_dir / "pred_3d" / f"test_{subject_id}_pred_softmax_3d.npz"
        gt_file = DATA_ROOT / "3d" / f"{subject_id}_3d.npz"
        
        if not pred_file.exists():
            print(f"  ⚠ 预测文件不存在: {pred_file.name}")
            continue
        if not gt_file.exists():
            print(f"  ⚠ GT文件不存在: {gt_file.name}")
            continue
        
        # 加载数据
        pred_data = np.load(pred_file)
        gt_data = np.load(gt_file)
        
        pred_probs = pred_data['pred_softmax_3d']  # (Z, X, Y, C)
        soft_ref = gt_data['proba_labels']  # (Z, X, Y, C)
        region_mask = gt_data['region_mask_lr']  # (Z, X, Y)
        
        # 生成 seg_labels（argmax of soft_ref，用于边界计算）
        seg_labels = np.argmax(soft_ref, axis=-1).astype(np.int32)  # (Z, X, Y)
        
        print(f"  Shapes: pred={pred_probs.shape}, ref={soft_ref.shape}, mask={region_mask.shape}")
        
        # 计算距离图
        dist_mm = distance_mm_from_boundary(seg_labels, voxel_spacing=VOXEL_SPACING)
        
        # 定义 Edge 和 Interior 掩膜（排除背景）
        foreground = region_mask > 0
        edge_mask = (dist_mm <= EDGE_THRESHOLD) & foreground
        interior_mask = (dist_mm >= INTERIOR_THRESHOLD) & foreground
        
        n_edge = edge_mask.sum()
        n_interior = interior_mask.sum()
        
        print(f"  Edge voxels: {n_edge}, Interior voxels: {n_interior}")
        
        if n_edge == 0 or n_interior == 0:
            print(f"  ⚠ Edge 或 Interior 为空，跳过")
            continue
        
        # 计算 Top-3 Coverage
        edge_cov = soft_topk_coverage(pred_probs, soft_ref, k=K_TOP, mask=edge_mask)
        interior_cov = soft_topk_coverage(pred_probs, soft_ref, k=K_TOP, mask=interior_mask)
        
        # 计算 soft-ECE
        edge_ece_val = soft_ece(pred_probs, soft_ref, n_bins=N_BINS_ECE, mask=edge_mask)
        interior_ece_val = soft_ece(pred_probs, soft_ref, n_bins=N_BINS_ECE, mask=interior_mask)
        
        # 存储结果（每个fold一个均值）
        results['edge_coverage'].append(edge_cov.mean())
        results['interior_coverage'].append(interior_cov.mean())
        results['edge_ece'].append(edge_ece_val)
        results['interior_ece'].append(interior_ece_val)
        
        print(f"  ✓ Top-3 cov: Edge={edge_cov.mean():.4f}, Interior={interior_cov.mean():.4f}")
        print(f"  ✓ Soft-ECE: Edge={edge_ece_val:.4f}, Interior={interior_ece_val:.4f}\n")
        
    except Exception as e:
        print(f"  ✗ 错误: {e}\n")
        continue

# 转为数组
for key in results:
    results[key] = np.array(results[key])

print("=" * 80)
print(f"完成！共处理 {len(results['edge_coverage'])} 个有效 folds")
print("=" * 80)